# DNS-over-TLS Functionality Test Plan

| Field | Value |
|---|---|
| **Document** | DoT functionality test plan (direct + globally forwarded) |
| **Scope** | Functional behavior of DNS-over-TLS resolution as a user would exercise it |
| **Out of scope** | Security testing (no fuzzing, no downgrade/renegotiation attacks, no load) |
| **Systems under test** | 1. The resolvers listed in `SERVERS`, queried **directly over DoT** 2. The same resolvers used as upstreams behind dnslab `global-forwarder-dot` forwarders (unbound, BIND, Knot Resolver) — **all traffic forwarded over DoT** |
| **Test zone** | `dnshl.us` — every `<type>.dnshl.us` name below is a **placeholder** to be provisioned with a real record before the record-coverage sections are meaningful |
| **Tooling** | dnspython (queries), Python `ssl` (certificate checks), dnslab (forwarder tier) |


## Executive summary

This is an **executable functionality test plan** for DNS-over-TLS resolution.
It exercises the configured resolvers directly over DoT and again behind
DoT-forwarding recursive servers (unbound, BIND, Knot Resolver via dnslab),
covering certificate validity, every provisioned record type, header flags,
EDNS (client subnet, extended errors, cookies), per-algorithm DNSSEC chains,
and multi-query TLS session behavior. Every check records purpose, expected
and observed outcomes with a PASS/FAIL/INFO/SKIP verdict; the *Results
summary & export* section at the end aggregates the run and writes the
CSV/JSON test record, and the *Report export* section renders a code-free
PDF of the whole document.

**Headline results** — fill in after each execution (numbers come from the
final summary section):

| Field | Value |
|---|---|
| Run date / operator | _(fill in)_ |
| Systems under test | _(fill in)_ |
| Overall verdict | _(PASS / FAIL from the summary cell)_ |
| Checks run / failed | _(fill in)_ |
| Notable findings | _(e.g. out-of-order pipelined responses, ECS subnet policy refusals, missing records to provision)_ |

## Test sections

| ID | Title | What it proves |
|---|---|---|
| TP-1 | TLS certificate verification | Each server presents a valid, hostname-matching, unexpired certificate chain |
| TP-2 | Record-type coverage | Every record type in the catalog resolves over DoT with the expected type in the answer |
| TP-3 | Header-flag behavior | Recursion, checking-disabled, and DNSSEC-OK flags behave as a client expects |
| TP-4 | EDNS behavior | Buffer sizes, absent EDNS, unknown options, and version negotiation don't break resolution |
| TP-4A | EDNS Client Subnet | ECS options with varied prefix/family/scope never break DoT; ECS-aware servers echo the option |
| TP-4B | Extended DNS Errors | SERVFAIL on a known-bogus name carries a useful EDE where supported |
| TP-4C | DNS COOKIE behavior | Queries with and without COOKIE options succeed; server cookie echo is recorded |
| TP-5 | DNSSEC deep review | Zones signed with each declared algorithm expose matching DNSKEY, DS, and RRSIGs, and validate (AD) |
| TP-6 | DoT session reuse & pipelining | Multiple queries per TLS session work — sequential reuse and true RFC 7766 pipelining |
| TP-7 | DoT forwarding | The full catalog and key variations behave identically through DoT-forwarding resolvers |
| — | Results summary & export | Aggregated verdicts, failure list, CSV/JSON artifacts (not a test section) |
| — | Report export | Code-free PDF report of this run (markdown + result tables only) |

## Result recording convention

Every individual check appends one row to a global `RESULTS` table:

| column | meaning |
|---|---|
| `section` / `test_id` | test-plan section and unique row id |
| `server` / `transport` | which target answered, and over what (`dot`, `do53`) |
| `qname` / `qtype` / `variation` | exactly what was asked, and how the query differed from a plain lookup |
| `expected` / `observed` | the acceptance criterion and what actually happened |
| `verdict` | **PASS** met expectation · **FAIL** did not · **INFO** behavior recorded, both outcomes acceptable · **SKIP** not executed |
| `notes` | anything a reader needs to interpret the row |

Run top-to-bottom. Each section is independent apart from TP-7, which requires
Docker (dnslab) and the servers in `SERVERS` to be reachable as upstreams.

## 0. Configuration

Edit this cell only. `SERVERS` are the direct systems under test **and** the
upstreams for the forwarded tier. `CA_FILE` supplies a private CA bundle for
upstream certificate validation (`None` = system trust store) — it is used for
direct queries, and passed to `dnslab.start(upstream_ca=...)` for the
forwarders.

In [ ]:
# ---- systems under test: DoT resolvers ----------------------------------
SERVERS = [
    {'ip': '1.1.1.1', 'port': 853, 'tls_hostname': 'one.one.one.one'},
    {'ip': '9.9.9.9', 'port': 853, 'tls_hostname': 'dns.quad9.net'},
    # {'ip': '10.0.0.53', 'port': 853, 'tls_hostname': 'rec.corp.example'},
]

# PEM bundle for validating the servers' certificates (None = system store).
CA_FILE = None
# CA_FILE = '/workspace/my-corp-ca.pem'

TEST_ZONE = 'dnshl.us'
TIMEOUT = 6

# ---- record-type catalog (TP-2 / TP-7) -----------------------------------
# (test_id, qname, qtype). Placeholders: provision each name with a real
# record of that type. Conventional owner forms are noted where they differ
# (SRV/TLSA normally live at underscored names - adjust once provisioned).
RECORD_TESTS = [
    ('A',          f'a.{TEST_ZONE}',          'A'),
    ('AAAA',       f'aaaa.{TEST_ZONE}',       'AAAA'),
    ('CNAME',      f'cname.{TEST_ZONE}',      'CNAME'),
    ('TXT',        f'txt.{TEST_ZONE}',        'TXT'),
    ('MX',         f'mx.{TEST_ZONE}',         'MX'),
    ('SRV',        f'srv.{TEST_ZONE}',        'SRV'),    # conventionally _svc._proto.name
    ('PTR',        f'ptr.{TEST_ZONE}',        'PTR'),
    ('NAPTR',      f'naptr.{TEST_ZONE}',      'NAPTR'),
    ('CAA',        f'caa.{TEST_ZONE}',        'CAA'),
    ('TLSA',       f'tlsa.{TEST_ZONE}',       'TLSA'),   # conventionally _443._tcp.name
    ('SSHFP',      f'sshfp.{TEST_ZONE}',      'SSHFP'),
    ('SVCB',       f'svcb.{TEST_ZONE}',       'SVCB'),
    ('HTTPS',      f'https.{TEST_ZONE}',      'HTTPS'),
    ('DNAME',      f'dname.{TEST_ZONE}',      'DNAME'),
    ('LOC',        f'loc.{TEST_ZONE}',        'LOC'),
    ('HINFO',      f'hinfo.{TEST_ZONE}',      'HINFO'),
    ('CERT',       f'cert.{TEST_ZONE}',       'CERT'),
    ('RP',         f'rp.{TEST_ZONE}',         'RP'),
    ('URI',        f'uri.{TEST_ZONE}',        'URI'),
    ('OPENPGPKEY', f'openpgpkey.{TEST_ZONE}', 'OPENPGPKEY'),
    ('SMIMEA',     f'smimea.{TEST_ZONE}',     'SMIMEA'),
    # zone-apex types
    ('SOA',        TEST_ZONE,                 'SOA'),
    ('NS',         TEST_ZONE,                 'NS'),
    ('DNSKEY',     TEST_ZONE,                 'DNSKEY'),
    ('DS',         TEST_ZONE,                 'DS'),     # served from the parent side
]

# names with known DNSSEC dispositions, used by TP-3/TP-6
SIGNED_NAME  = 'internetsociety.org'   # validly signed: expect AD from validators
BOGUS_NAME   = 'dnssec-failed.org'     # deliberately bogus: expect SERVFAIL (+EDE)
NEUTRAL_NAME = 'example.com'

# ECS test name (TP-4E): point at a name served by an ECS-capable
# authoritative once provisioned, e.g. f'ecs.{TEST_ZONE}'
ECS_QNAME = NEUTRAL_NAME

# EDNS advertised buffer sizes exercised by TP-4 (None = no EDNS at all)
EDNS_BUFSIZES = [None, 512, 1232, 4096]

# ---- forwarded tier (TP-7): dnslab servers, all forwarding over DoT ------
FORWARDERS = [
    ('unbound',       'global-forwarder-dot'),
    ('bind',          'global-forwarder-dot'),
    ('knot-resolver', 'global-forwarder-dot'),
]
FORCE_RESTART = False   # True: recreate forwarders even if running unchanged
print(f'{len(SERVERS)} direct server(s), {len(RECORD_TESTS)} record tests, '
      f'{len(FORWARDERS)} forwarder(s) configured')

## 1. Test harness

`probe()` issues one query over `dot` (TLS-validated) or `do53` and returns a
structured observation — rcode, header flags, EDNS version/payload echoed,
EDE options, COOKIE echo, answer types, RRSIG presence, latency, or the
transport error. It never raises: failures become part of the record.
`tls_cert_report()` captures the certificate a server presents.
`record()` appends one verdict row to `RESULTS`.

In [ ]:
import os, socket, ssl, time
from datetime import datetime, timezone

import dns.edns, dns.flags, dns.message, dns.query, dns.rcode, dns.rdatatype
import pandas as pd
from IPython.display import display

RESULTS = []
COOKIE_T = getattr(dns.edns, 'COOKIE', 10)
EDE_T = getattr(dns.edns, 'EDE', 15)
ECS_T = getattr(dns.edns, 'ECS', 8)


def record(section, test_id, server, transport, qname, qtype, variation,
           expected, observed, verdict, notes=''):
    row = dict(section=section, test_id=test_id, server=server,
               transport=transport, qname=qname, qtype=qtype,
               variation=variation, expected=expected, observed=observed,
               verdict=verdict, notes=notes)
    RESULTS.append(row)
    return row


def _ssl_ctx(ca):
    return ssl.create_default_context(cafile=ca) if ca else ssl.create_default_context()


def probe(srv, qname, qtype='A', *, transport='dot', rd=True, cd=False,
          do=False, edns=True, payload=1232, version=0, cookie=None,
          extra_options=None, ca=None, timeout=None):
    """One query -> observation dict. Never raises; errors are recorded."""
    out = dict(rcode=None, flags='', edns=None, payload=None, ede=[],
               cookie_echoed=None, ecs=None, dnskey_algs=[], dnskey_flags=[],
               ds_algs=[], ds_digests=[], rrsig_algs=[], answers=0, answer_types=[], rrsig=False,
               time_ms=None, error=None)
    try:
        opts = list(extra_options or [])
        if cookie is not None:
            opts.append(dns.edns.GenericOption(COOKIE_T, cookie))
        if edns:
            q = dns.message.make_query(qname, qtype, use_edns=version,
                                       payload=payload, want_dnssec=do,
                                       options=opts or None)
        else:
            q = dns.message.make_query(qname, qtype)
        if not rd:
            q.flags &= ~dns.flags.RD
        if cd:
            q.flags |= dns.flags.CD
        t0 = time.monotonic()
        if transport == 'dot':
            use_ca = ca if ca is not None else srv.get('ca_file', CA_FILE)
            r = dns.query.tls(q, srv['ip'], port=srv.get('port', 853),
                              timeout=timeout or TIMEOUT,
                              server_hostname=srv['tls_hostname'],
                              ssl_context=_ssl_ctx(use_ca))
        else:
            r, _tcp = dns.query.udp_with_fallback(
                q, srv['ip'], port=srv.get('port_do53', 53),
                timeout=timeout or TIMEOUT)
        out['time_ms'] = round((time.monotonic() - t0) * 1000, 1)
        out['rcode'] = dns.rcode.to_text(r.rcode())
        out['flags'] = dns.flags.to_text(r.flags)
        out['edns'] = r.edns                       # -1 = no OPT in response
        out['payload'] = r.payload if r.edns >= 0 else None
        for o in (r.options or []):
            if o.otype == EDE_T:
                code_ = int(getattr(o, 'code', -1))
                text = getattr(o, 'text', '') or ''
                out['ede'].append(f'{code_}:{text}'.rstrip(':'))
            elif o.otype == COOKIE_T:
                server_part = getattr(o, 'server', None)
                data = getattr(o, 'data', b'')
                out['cookie_echoed'] = bool(server_part) or len(data) > 8
            elif o.otype == ECS_T:
                addr = getattr(o, 'address', None)
                if addr is not None:
                    out['ecs'] = (f"{addr}/{getattr(o, 'srclen', '?')} "
                                  f"scope {getattr(o, 'scopelen', '?')}")
                else:
                    out['ecs'] = 'present'
        out['answers'] = sum(len(rrset) for rrset in r.answer)
        out['answer_types'] = sorted({dns.rdatatype.to_text(rrset.rdtype)
                                      for rrset in r.answer})
        out['rrsig'] = 'RRSIG' in out['answer_types']
        for rrset in r.answer:
            if rrset.rdtype == dns.rdatatype.DNSKEY:
                out['dnskey_algs'] = sorted({*out['dnskey_algs'],
                                             *(int(rd.algorithm) for rd in rrset)})
                out['dnskey_flags'] = sorted({*out['dnskey_flags'],
                                              *(int(rd.flags) for rd in rrset)})
            elif rrset.rdtype == dns.rdatatype.DS:
                out['ds_algs'] = sorted({*out['ds_algs'],
                                         *(int(rd.algorithm) for rd in rrset)})
                out['ds_digests'] = sorted({*out['ds_digests'],
                                            *(int(rd.digest_type) for rd in rrset)})
            elif rrset.rdtype == dns.rdatatype.RRSIG:
                out['rrsig_algs'] = sorted({*out['rrsig_algs'],
                                            *(int(rd.algorithm) for rd in rrset)})
    except Exception as e:  # noqa: BLE001 — observation, not control flow
        out['error'] = f'{type(e).__name__}: {e}'
    return out


def tls_cert_report(srv, ca=None):
    """TLS handshake + certificate details for one server."""
    info = dict(ok=False, tls_version=None, cipher=None, subject_cn=None,
                issuer=None, san=[], not_after=None, days_left=None, error=None)
    try:
        use_ca = ca if ca is not None else srv.get('ca_file', CA_FILE)
        ctx = _ssl_ctx(use_ca)
        with socket.create_connection((srv['ip'], srv.get('port', 853)),
                                      timeout=TIMEOUT) as s:
            with ctx.wrap_socket(s, server_hostname=srv['tls_hostname']) as tls:
                cert = tls.getpeercert()
                info['tls_version'] = tls.version()
                info['cipher'] = tls.cipher()[0]
        subj = dict(x[0] for x in cert.get('subject', ()))
        iss = dict(x[0] for x in cert.get('issuer', ()))
        info['subject_cn'] = subj.get('commonName')
        info['issuer'] = iss.get('commonName') or iss.get('organizationName')
        info['san'] = [v for k, v in cert.get('subjectAltName', ()) if k == 'DNS'][:8]
        info['not_after'] = cert.get('notAfter')
        info['days_left'] = int((ssl.cert_time_to_seconds(cert['notAfter'])
                                 - time.time()) // 86400)
        info['ok'] = True
    except Exception as e:  # noqa: BLE001
        info['error'] = f'{type(e).__name__}: {e}'
    return info


def show(rows):
    df = pd.DataFrame(rows)
    display(df)
    return df


def obs_str(o):
    """Compact human-readable observation for the results table."""
    if o['error']:
        return o['error']
    bits = [o['rcode'], f"flags[{o['flags']}]", f"ans={o['answers']}"]
    if o['answer_types']:
        bits.append('+'.join(o['answer_types']))
    if o['edns'] is not None and o['edns'] >= 0:
        bits.append(f"edns{o['edns']}/{o['payload']}")
    else:
        bits.append('no-edns')
    if o['ede']:
        bits.append('EDE[' + '; '.join(o['ede']) + ']')
    if o['cookie_echoed'] is not None:
        bits.append('cookie-echo=' + str(o['cookie_echoed']))
    if o['ecs'] is not None:
        bits.append(f"ecs[{o['ecs']}]")
    for key, tag in (('dnskey_algs', 'dnskey-alg'), ('ds_algs', 'ds-alg'),
                     ('rrsig_algs', 'rrsig-alg'), ('ds_digests', 'ds-digest'),
                     ('dnskey_flags', 'dnskey-flags')):
        if o.get(key):
            bits.append(f"{tag}={','.join(map(str, o[key]))}")
    return ' '.join(bits)


print('harness ready,', datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%MZ'))

## TP-1 — TLS certificate verification

| | |
|---|---|
| **Purpose** | Confirm every server in `SERVERS` completes a TLS handshake with certificate validation ON, presents a certificate whose SAN matches `tls_hostname`, chains to the configured trust store (`CA_FILE` or system), and is not close to expiry. |
| **Procedure** | Open a validated TLS connection to `ip:port` with SNI = `tls_hostname`; capture negotiated TLS version, cipher, subject CN, issuer, SANs, expiry. |
| **Expected** | Handshake succeeds (validation implies hostname + chain OK); TLS ≥ 1.2; ≥ 15 days to expiry. |
| **Verdict rule** | PASS all conditions · FAIL handshake/validation error or < 15 days left. |

In [ ]:
SEC = 'TP-1 tls-cert'
rows = []
for i, srv in enumerate(SERVERS, 1):
    label = f"{srv['tls_hostname']} ({srv['ip']})"
    c = tls_cert_report(srv)
    if not c['ok']:
        verdict, obs = 'FAIL', c['error']
    elif c['days_left'] is not None and c['days_left'] < 15:
        verdict = 'FAIL'
        obs = f"valid handshake but only {c['days_left']}d to expiry"
    else:
        verdict = 'PASS'
        obs = (f"{c['tls_version']} {c['cipher']}; CN={c['subject_cn']}; "
               f"issuer={c['issuer']}; expires {c['not_after']} "
               f"({c['days_left']}d)")
    rows.append(record(SEC, f'TP-1.{i}', label, 'dot', '-', '-',
                       'validated handshake + cert inspection',
                       'handshake OK, hostname+chain valid, >=15d to expiry',
                       obs, verdict, notes=' '.join(c['san'])))
show(rows);

## TP-2 — Record-type coverage, direct DoT

| | |
|---|---|
| **Purpose** | Confirm each record type in the catalog **exists and resolves** over DoT from every server. This is the placeholder-driven section: each `<type>.dnshl.us` must be provisioned with a real record of that type. |
| **Procedure** | For each (qname, qtype) and each server: DoT query with RD=1, DO=1. |
| **Expected** | NOERROR with at least one answer of the queried type (a CNAME in the answer chain is acceptable). |
| **Verdict rule** | PASS as expected · FAIL NXDOMAIN (name absent), NODATA (type absent), unexpected rcode, or transport error. |

In [ ]:
SEC = 'TP-2 record types (direct)'
rows = []
for srv in SERVERS:
    label = srv['tls_hostname']
    for tid, qname, qtype in RECORD_TESTS:
        o = probe(srv, qname, qtype, do=True)
        if o['error']:
            verdict = 'FAIL'
        elif o['rcode'] == 'NOERROR' and (qtype in o['answer_types']
                                          or 'CNAME' in o['answer_types']):
            verdict = 'PASS'
        elif o['rcode'] == 'NOERROR':
            verdict = 'FAIL'   # NODATA: name exists, type missing
        else:
            verdict = 'FAIL'   # NXDOMAIN or other
        rows.append(record(SEC, f'TP-2.{tid}.{label}', label, 'dot',
                           qname, qtype, 'RD=1 DO=1',
                           f'NOERROR with {qtype} (or CNAME) in answer',
                           obs_str(o), verdict))
df = show(rows)
print(df.verdict.value_counts().to_dict())

## TP-3 — Header-flag behavior: RD, CD, DO (+dnssec)

| | |
|---|---|
| **Purpose** | Confirm the flags a client can set produce the behavior a user relies on. |
| **Procedure & expectations** | **RD=1** on `NEUTRAL_NAME`: NOERROR with answers, RA set. **RD=0** (nord): a recursive server must still *respond* — REFUSED or an empty/ referral answer are both acceptable; verdict INFO records which. **DO=1** on `SIGNED_NAME`: AD flag set (server validates) and/or RRSIGs returned. **DO=1** on `BOGUS_NAME`: SERVFAIL from a validating server. **CD=1** on `BOGUS_NAME`: answer returned despite bogus DNSSEC (validation disabled) — NOERROR expected from validators. |
| **Verdict rule** | As per each row's `expected`; `nord` rows are INFO (behavior legitimately varies). |

In [ ]:
SEC = 'TP-3 flags'
rows = []
for srv in SERVERS:
    label = srv['tls_hostname']

    o = probe(srv, NEUTRAL_NAME, 'A')
    ok = (not o['error']) and o['rcode'] == 'NOERROR' and o['answers'] > 0 \
         and 'RA' in o['flags']
    rows.append(record(SEC, f'TP-3.rd.{label}', label, 'dot', NEUTRAL_NAME,
                       'A', 'RD=1', 'NOERROR, answers, RA set', obs_str(o),
                       'PASS' if ok else 'FAIL'))

    o = probe(srv, NEUTRAL_NAME, 'A', rd=False)
    responded = not o['error']
    rows.append(record(SEC, f'TP-3.nord.{label}', label, 'dot', NEUTRAL_NAME,
                       'A', 'RD=0 (nord)',
                       'server responds (REFUSED or empty both acceptable)',
                       obs_str(o), 'INFO' if responded else 'FAIL',
                       notes='recursive servers may refuse iterative queries'))

    o = probe(srv, SIGNED_NAME, 'A', do=True)
    ok = (not o['error']) and o['rcode'] == 'NOERROR' \
         and ('AD' in o['flags'] or o['rrsig'])
    rows.append(record(SEC, f'TP-3.do.{label}', label, 'dot', SIGNED_NAME,
                       'A', 'DO=1 (+dnssec)',
                       'NOERROR with AD set and/or RRSIGs in answer',
                       obs_str(o), 'PASS' if ok else 'FAIL'))

    o = probe(srv, BOGUS_NAME, 'A', do=True)
    ok = (not o['error']) and o['rcode'] == 'SERVFAIL'
    rows.append(record(SEC, f'TP-3.bogus.{label}', label, 'dot', BOGUS_NAME,
                       'A', 'DO=1 on bogus zone',
                       'SERVFAIL (validation failure)', obs_str(o),
                       'PASS' if ok else 'FAIL'))

    o = probe(srv, BOGUS_NAME, 'A', do=True, cd=True)
    ok = (not o['error']) and o['rcode'] == 'NOERROR' and o['answers'] > 0
    rows.append(record(SEC, f'TP-3.cd.{label}', label, 'dot', BOGUS_NAME,
                       'A', 'CD=1 on bogus zone',
                       'NOERROR with answers (validation bypassed)',
                       obs_str(o), 'PASS' if ok else 'FAIL'))
show(rows);

## TP-4 — EDNS behavior: buffer sizes, no-EDNS, unknown option, version

| | |
|---|---|
| **Purpose** | Confirm EDNS negotiation over DoT never breaks resolution. Over TLS (TCP) the advertised UDP buffer size is largely moot — the point is that *no advertised size, including none at all, causes a failure*. |
| **Procedure & expectations** | Query `NEUTRAL_NAME` A with: no EDNS (plain DNS): NOERROR, response may or may not carry OPT. EDNS0 payload 512 / 1232 / 4096: NOERROR, response OPT version 0, no TC. EDNS0 + unknown option (code 65280, experimental range): NOERROR, option NOT echoed back. EDNS **version 1**: RFC 6891 requires **BADVERS** — servers answering NOERROR get INFO (tolerant behavior, worth knowing, not a user-facing defect). |
| **Verdict rule** | As per row; version-1 row: PASS on BADVERS, INFO on NOERROR, FAIL on error/timeouts. |

In [ ]:
SEC = 'TP-4 edns'
rows = []
for srv in SERVERS:
    label = srv['tls_hostname']

    for size in EDNS_BUFSIZES:
        if size is None:
            o = probe(srv, NEUTRAL_NAME, 'A', edns=False)
            var, exp = 'no EDNS', 'NOERROR with answers'
        else:
            o = probe(srv, NEUTRAL_NAME, 'A', payload=size)
            var, exp = f'EDNS0 payload={size}', 'NOERROR, OPT v0, no TC'
        ok = (not o['error']) and o['rcode'] == 'NOERROR' \
             and o['answers'] > 0 and 'TC' not in o['flags']
        if size is not None:
            ok = ok and o['edns'] == 0
        rows.append(record(SEC, f'TP-4.buf{size}.{label}', label, 'dot',
                           NEUTRAL_NAME, 'A', var, exp, obs_str(o),
                           'PASS' if ok else 'FAIL'))

    unknown = dns.edns.GenericOption(65280, b'dnslab-functional-test')
    o = probe(srv, NEUTRAL_NAME, 'A', extra_options=[unknown])
    ok = (not o['error']) and o['rcode'] == 'NOERROR' and o['answers'] > 0
    rows.append(record(SEC, f'TP-4.unkopt.{label}', label, 'dot',
                       NEUTRAL_NAME, 'A', 'EDNS option 65280 (unknown)',
                       'NOERROR; unknown option ignored, not echoed',
                       obs_str(o), 'PASS' if ok else 'FAIL'))

    o = probe(srv, NEUTRAL_NAME, 'A', version=1)
    if o['error']:
        verdict = 'FAIL'
    elif o['rcode'] == 'BADVERS':
        verdict = 'PASS'
    elif o['rcode'] == 'NOERROR':
        verdict = 'INFO'
    else:
        verdict = 'FAIL'
    rows.append(record(SEC, f'TP-4.v1.{label}', label, 'dot', NEUTRAL_NAME,
                       'A', 'EDNS version 1',
                       'BADVERS per RFC 6891 (NOERROR tolerated: INFO)',
                       obs_str(o), verdict))
show(rows);

## TP-4A — EDNS Client Subnet (RFC 7871)

| | |
|---|---|
| **Purpose** | Confirm that sending an ECS option over DoT never breaks resolution, across IPv4/IPv6 families, source prefix lengths (/16, /24, /32), the 0.0.0.0/0 opt-out form, and an illegal nonzero scope. Record whether the recursive server is **ECS-aware** — i.e. echoes an ECS option (with its chosen scope) in the response. Verifying that answers actually vary by subnet requires an ECS-capable authoritative behind `ECS_QNAME`; that provisioning is noted per row but only the echo is asserted here. |
| **Procedure** | Query `ECS_QNAME` A over DoT with each ECS variation as an EDNS option. |
| **Expected** | Scope-0 rows: NOERROR; an ECS-aware server echoes the option. Opt-out row: NOERROR; echo (if any) with scope 0 and no client subnet used. Nonzero-scope row: RFC 7871 §6 requires scope 0 in queries — FORMERR and tolerant NOERROR are both legitimate server reactions. |
| **Verdict rule** | Scope-0/opt-out: PASS NOERROR + ECS echoed · INFO NOERROR without echo (query unharmed, server strips/ignores ECS) · INFO REFUSED **with** ECS echoed (subnet-policy filtering: the default test subnets are documentation ranges — TEST-NET-2, 2001:db8:: — which some ECS-aware resolvers, e.g. dns.google, reject as unroutable; substitute prefixes you are authorized to test with) · FAIL transport error or other rcode. Nonzero scope: INFO on any response · FAIL only on transport error. |

In [ ]:
SEC = 'TP-4A ecs'
# NB: subnets below are documentation ranges (safe to publish). Some
# ECS-aware resolvers refuse them as unroutable (observed: dns.google
# REFUSED TEST-NET-2 /24 but accepted 198.51.0.0/16) - substitute real
# prefixes you are authorized to test with for meaningful scope results.
ECS_TESTS = [
    ('v4 /24 scope 0',        dns.edns.ECSOption('198.51.100.0', 24, 0),  'normal'),
    ('v4 /32 scope 0',        dns.edns.ECSOption('198.51.100.7', 32, 0),  'normal'),
    ('v4 /16 scope 0',        dns.edns.ECSOption('198.51.0.0', 16, 0),    'normal'),
    ('v4 0.0.0.0/0 opt-out',  dns.edns.ECSOption('0.0.0.0', 0, 0),        'optout'),
    ('v6 /56 scope 0',        dns.edns.ECSOption('2001:db8:1234::', 56, 0), 'normal'),
    ('v4 /24 scope 24 (illegal in query)',
                              dns.edns.ECSOption('198.51.100.0', 24, 24), 'badscope'),
]
rows = []
for srv in SERVERS:
    label = srv['tls_hostname']
    for i, (var, opt, kind) in enumerate(ECS_TESTS, 1):
        o = probe(srv, ECS_QNAME, 'A', extra_options=[opt])
        note = '' if o['ecs'] else 'no ECS echo - server strips/ignores ECS'
        if o['error']:
            verdict = 'FAIL'
        elif kind == 'badscope':
            verdict = 'INFO'
        elif o['rcode'] == 'NOERROR' and o['ecs']:
            verdict = 'PASS'
        elif o['rcode'] == 'NOERROR':
            verdict = 'INFO'
        elif o['rcode'] == 'REFUSED' and o['ecs']:
            verdict = 'INFO'
            note = ('ECS-aware server refused this subnet (documentation-'
                    'range filtering) - retry with a routable prefix')
        else:
            verdict = 'FAIL'
        if kind == 'badscope':
            exp = 'response received (FORMERR or NOERROR both legitimate)'
        else:
            exp = 'NOERROR; ECS option echoed by an ECS-aware server'
        rows.append(record(SEC, f'TP-4A.{i}.{label}', label, 'dot', ECS_QNAME,
                           'A', f'ECS {var}', exp, obs_str(o), verdict,
                           notes=note))
show(rows);

## TP-4B — Extended DNS Errors (RFC 8914)

| | |
|---|---|
| **Purpose** | When resolution fails, a user-facing resolver should say *why*. Confirm a DNSSEC-bogus name produces SERVFAIL, and record whether an EDE option (e.g. 6 "DNSSEC Bogus", 9 "DNSKEY Missing") accompanies it. |
| **Procedure** | Query `BOGUS_NAME` A with DO=1 over DoT. |
| **Verdict rule** | PASS SERVFAIL **with** at least one EDE · INFO SERVFAIL without EDE (correct outcome, less diagnosable) · FAIL any other rcode or transport error. |

In [ ]:
SEC = 'TP-4B ede'
rows = []
for srv in SERVERS:
    label = srv['tls_hostname']
    o = probe(srv, BOGUS_NAME, 'A', do=True)
    if o['error'] or o['rcode'] != 'SERVFAIL':
        verdict = 'FAIL'
    elif o['ede']:
        verdict = 'PASS'
    else:
        verdict = 'INFO'
    rows.append(record(SEC, f'TP-4B.{label}', label, 'dot', BOGUS_NAME, 'A',
                       'DO=1 on bogus zone',
                       'SERVFAIL with EDE explaining the failure',
                       obs_str(o), verdict))
show(rows);

## TP-4C — DNS COOKIE behavior (RFC 7873)

| | |
|---|---|
| **Purpose** | Confirm that sending a COOKIE option never breaks resolution over DoT, and record whether the server participates in the cookie exchange. Cookies exist to protect *UDP*; over an authenticated TLS channel many servers deliberately skip them, so absence of an echo is **acceptable** (INFO). |
| **Procedure** | Query `NEUTRAL_NAME` A three ways: without any COOKIE option (baseline); with a fresh 8-byte client cookie; with a malformed (4-byte) client cookie. |
| **Expected** | Baseline and valid-cookie queries: NOERROR with answers; server cookie echo recorded either way. Malformed cookie: server responds (FORMERR or NOERROR both acceptable — INFO) rather than dropping the connection. |

In [ ]:
SEC = 'TP-4C cookies'
rows = []
for srv in SERVERS:
    label = srv['tls_hostname']

    o = probe(srv, NEUTRAL_NAME, 'A')
    ok = (not o['error']) and o['rcode'] == 'NOERROR' and o['answers'] > 0
    rows.append(record(SEC, f'TP-4C.none.{label}', label, 'dot', NEUTRAL_NAME,
                       'A', 'no COOKIE option', 'NOERROR with answers',
                       obs_str(o), 'PASS' if ok else 'FAIL'))

    o = probe(srv, NEUTRAL_NAME, 'A', cookie=os.urandom(8))
    ok = (not o['error']) and o['rcode'] == 'NOERROR' and o['answers'] > 0
    echo = o['cookie_echoed']
    rows.append(record(SEC, f'TP-4C.cookie.{label}', label, 'dot',
                       NEUTRAL_NAME, 'A', 'valid 8-byte client COOKIE',
                       'NOERROR with answers; echo recorded', obs_str(o),
                       'PASS' if ok else 'FAIL',
                       notes=f'server cookie echoed: {echo} '
                             '(no echo is acceptable over TLS)'))

    o = probe(srv, NEUTRAL_NAME, 'A', cookie=os.urandom(4))
    responded = not o['error']
    rows.append(record(SEC, f'TP-4C.malformed.{label}', label, 'dot',
                       NEUTRAL_NAME, 'A', 'malformed 4-byte COOKIE',
                       'server responds (FORMERR or NOERROR acceptable)',
                       obs_str(o), 'INFO' if responded else 'FAIL'))
show(rows);

## TP-5 — DNSSEC deep review (per-algorithm)

| | |
|---|---|
| **Purpose** | For each provided zone and its declared DNSSEC algorithm, confirm the full signing chain is visible and functional over DoT: the zone publishes DNSKEYs of that algorithm, the parent publishes a matching DS, and answers validate (AD) with RRSIGs of that algorithm. `DNSSEC_DOMAINS` is the input — provision each placeholder zone signed with the listed algorithm; `cloudflare.com`/13 is a live control proving the checker works today. |
| **Procedure** | Per domain × server, three DoT queries with DO=1: `DNSKEY` (zone keys), `DS` (delegation, served from the parent side), `A` (validated resolution). |
| **Expected** | DNSKEY: NOERROR with at least one DNSKEY of the declared algorithm (key flags recorded — 257 KSK / 256 ZSK). DS: NOERROR with a DS whose algorithm field matches (digest types recorded). A: NOERROR with **AD** set and an RRSIG of the declared algorithm. |
| **Verdict rule** | PASS on match · FAIL on NXDOMAIN/NODATA (zone or delegation not provisioned), algorithm mismatch (wrong-key detection is the point of this section), missing DS (insecure delegation), or missing AD. |

In [ ]:
SEC = 'TP-5 dnssec'
import dns.dnssec

# ---- INPUT: zones and the DNSSEC algorithm each is signed with -----------
# (domain, algorithm number). Placeholders: sign each <alg>.dnshl.us zone
# with that algorithm and delegate it (DS in dnshl.us) before scoring this
# section. cloudflare.com is a live control for the checker itself.
DNSSEC_DOMAINS = [
    (f'rsasha256.{TEST_ZONE}', 8),    # RSASHA256
    (f'rsasha512.{TEST_ZONE}', 10),   # RSASHA512
    (f'ecdsa256.{TEST_ZONE}', 13),    # ECDSAP256SHA256
    (f'ecdsa384.{TEST_ZONE}', 14),    # ECDSAP384SHA384
    (f'ed25519.{TEST_ZONE}', 15),     # ED25519
    (f'ed448.{TEST_ZONE}', 16),       # ED448
    ('cloudflare.com', 13),           # live control (ECDSAP256SHA256)
]

rows = []
for srv in SERVERS:
    label = srv['tls_hostname']
    for domain, alg in DNSSEC_DOMAINS:
        alg_txt = dns.dnssec.algorithm_to_text(alg)

        o = probe(srv, domain, 'DNSKEY', do=True)
        if o['error'] or o['rcode'] != 'NOERROR' or not o['dnskey_algs']:
            verdict = 'FAIL'
        else:
            verdict = 'PASS' if alg in o['dnskey_algs'] else 'FAIL'
        flags = {257: 'KSK', 256: 'ZSK'}
        rows.append(record(SEC, f'TP-5.dnskey.{domain}.{label}', label, 'dot',
                           domain, 'DNSKEY', f'declared alg {alg} ({alg_txt})',
                           f'NOERROR with DNSKEY alg {alg}', obs_str(o), verdict,
                           notes='keys: ' + ','.join(flags.get(f, str(f))
                                 for f in o['dnskey_flags'])))

        o = probe(srv, domain, 'DS', do=True)
        if o['error'] or o['rcode'] != 'NOERROR':
            verdict, note = 'FAIL', ''
        elif not o['ds_algs']:
            verdict, note = 'FAIL', 'no DS - insecure delegation (not provisioned?)'
        elif alg in o['ds_algs']:
            verdict, note = 'PASS', ''
        else:
            verdict, note = 'FAIL', 'DS present but algorithm mismatch'
        rows.append(record(SEC, f'TP-5.ds.{domain}.{label}', label, 'dot',
                           domain, 'DS', f'declared alg {alg} ({alg_txt})',
                           f'NOERROR with DS alg {alg} from the parent',
                           obs_str(o), verdict, notes=note))

        o = probe(srv, domain, 'A', do=True)
        if o['error'] or o['rcode'] != 'NOERROR':
            verdict, note = 'FAIL', ''
        elif 'AD' not in o['flags']:
            verdict, note = 'FAIL', 'no AD - resolver did not validate'
        elif alg in o['rrsig_algs']:
            verdict, note = 'PASS', ''
        elif not o['rrsig_algs']:
            verdict, note = 'FAIL', 'AD set but no RRSIG in answer'
        else:
            verdict, note = 'FAIL', 'RRSIG algorithm mismatch'
        rows.append(record(SEC, f'TP-5.valid.{domain}.{label}', label, 'dot',
                           domain, 'A', f'DO=1, declared alg {alg} ({alg_txt})',
                           f'NOERROR, AD set, RRSIG alg {alg}', obs_str(o),
                           verdict, notes=note))
df = show(rows)
print(df.verdict.value_counts().to_dict())

## TP-6 — DoT session reuse & query pipelining (RFC 7766/7858)

| | |
|---|---|
| **Purpose** | A DoT client that opens a fresh TLS session per query wastes a handshake every time; well-behaved servers support many queries per session, including **pipelining** (multiple queries written before any response is read) with possibly out-of-order responses. Confirm both work — dnspython supports them via `dns.query.tls(sock=...)` for sequential reuse and `dns.query.send_tcp()`/`receive_tcp()` for true pipelining. |
| **Procedure** | Per server, on ONE TLS session each: (a) 1 query (baseline); (b) 5 queries sequentially reusing the session; (c) 20 queries pipelined — all written back-to-back, then all responses read and matched to queries by message id; (d) INFO timing row: 20 pipelined on one session vs 20 queries each on a fresh session. Queries rotate qtypes (A/AAAA/MX/TXT/NS) over `NEUTRAL_NAME` so every response should be NOERROR. |
| **Expected** | All queries on the shared session answered NOERROR with no reconnect; every pipelined response matches a sent query id (out-of-order arrival is legitimate and recorded); pipelining materially faster than fresh-session-per-query. |
| **Verdict rule** | (a)-(c): PASS all n answered NOERROR · FAIL premature close, unmatched response, or non-NOERROR. (d): INFO (timing observation). |

In [ ]:
SEC = 'TP-6 pipeline'
PIPELINE_QTYPES = ['A', 'AAAA', 'MX', 'TXT', 'NS']


def dot_session(srv):
    """One validated TLS session to srv (caller closes)."""
    ctx = _ssl_ctx(srv.get('ca_file', CA_FILE))
    raw = socket.create_connection((srv['ip'], srv.get('port', 853)),
                                   timeout=TIMEOUT)
    tls = ctx.wrap_socket(raw, server_hostname=srv['tls_hostname'])
    tls.settimeout(TIMEOUT)
    return tls


def _mk(i):
    return dns.message.make_query(NEUTRAL_NAME,
                                  PIPELINE_QTYPES[i % len(PIPELINE_QTYPES)],
                                  use_edns=0)


def run_sequential(srv, n):
    """n queries one-at-a-time over a single TLS session."""
    rcodes, t0 = [], time.monotonic()
    with dot_session(srv) as tls:
        for i in range(n):
            r = dns.query.tls(_mk(i), None, sock=tls, timeout=TIMEOUT)
            rcodes.append(dns.rcode.to_text(r.rcode()))
    return rcodes, (time.monotonic() - t0) * 1000


def run_pipelined(srv, n):
    """n queries written back-to-back, then all responses read (RFC 7766)."""
    sent, rcodes, in_order = {}, [], True
    t0 = time.monotonic()
    with dot_session(srv) as tls:
        for i in range(n):
            q = _mk(i)
            dns.query.send_tcp(tls, q)
            sent[q.id] = q
        send_order = list(sent)
        while len(rcodes) < n:
            r, _ = dns.query.receive_tcp(tls, time.time() + TIMEOUT)
            if r.id not in sent or not sent[r.id].is_response(r):
                raise RuntimeError('response did not match any pipelined query')
            if r.id != send_order[len(rcodes)]:
                in_order = False
            rcodes.append(dns.rcode.to_text(r.rcode()))
    return rcodes, (time.monotonic() - t0) * 1000, in_order


def run_fresh(srv, n):
    """n queries, each on its own brand-new TLS session (the anti-pattern)."""
    t0 = time.monotonic()
    rcodes = []
    for i in range(n):
        with dot_session(srv) as tls:
            r = dns.query.tls(_mk(i), None, sock=tls, timeout=TIMEOUT)
            rcodes.append(dns.rcode.to_text(r.rcode()))
    return rcodes, (time.monotonic() - t0) * 1000


rows = []
for srv in SERVERS:
    label = srv['tls_hostname']

    for n, runner, var in ((1, run_sequential, '1 query, fresh session'),
                           (5, run_sequential, '5 queries, one session (sequential reuse)')):
        try:
            rcodes, ms = runner(srv, n)
            ok = len(rcodes) == n and all(rc == 'NOERROR' for rc in rcodes)
            obs = f'{len(rcodes)}/{n} answered {"+".join(sorted(set(rcodes)))} in {ms:.0f}ms'
            verdict = 'PASS' if ok else 'FAIL'
        except Exception as e:  # noqa: BLE001
            obs, verdict = f'{type(e).__name__}: {e}', 'FAIL'
        rows.append(record(SEC, f'TP-6.seq{n}.{label}', label, 'dot',
                           NEUTRAL_NAME, '+'.join(PIPELINE_QTYPES[:n]) if n <= 5
                           else 'rotating', var,
                           f'all {n} answered NOERROR on one session',
                           obs, verdict))

    try:
        rcodes, pipe_ms, in_order = run_pipelined(srv, 20)
        ok = len(rcodes) == 20 and all(rc == 'NOERROR' for rc in rcodes)
        obs = (f'20/20 answered {"+".join(sorted(set(rcodes)))} in {pipe_ms:.0f}ms, '
               f'responses {"in order" if in_order else "OUT of order (legitimate per RFC 7766)"}')
        verdict = 'PASS' if ok else 'FAIL'
    except Exception as e:  # noqa: BLE001
        obs, verdict, pipe_ms = f'{type(e).__name__}: {e}', 'FAIL', None
    rows.append(record(SEC, f'TP-6.pipe20.{label}', label, 'dot',
                       NEUTRAL_NAME, 'rotating', '20 queries pipelined, one session',
                       'all 20 answered NOERROR; id-matched; order recorded',
                       obs, verdict))

    try:
        _, fresh_ms = run_fresh(srv, 20)
        if pipe_ms:
            obs = (f'pipelined 20: {pipe_ms:.0f}ms vs 20 fresh sessions: '
                   f'{fresh_ms:.0f}ms ({fresh_ms / pipe_ms:.1f}x slower)')
        else:
            obs = f'20 fresh sessions: {fresh_ms:.0f}ms (pipeline failed, no ratio)'
        verdict = 'INFO'
    except Exception as e:  # noqa: BLE001
        obs, verdict = f'{type(e).__name__}: {e}', 'FAIL'
    rows.append(record(SEC, f'TP-6.timing.{label}', label, 'dot',
                       NEUTRAL_NAME, 'rotating', 'timing: pipeline vs fresh sessions',
                       'pipelining materially faster (observation)', obs, verdict))
show(rows);

## TP-7 — DoT forwarding (global forwarder configuration, dnslab)

| | |
|---|---|
| **Purpose** | Prove the same functionality holds when clients sit behind a resolver that forwards **all** traffic over DoT — the `global-forwarder-dot` profile on unbound, BIND, and Knot Resolver, using `SERVERS` as upstreams (validated with `CA_FILE` when set, else the system store). |
| **Preconditions** | Docker available to dnslab; `SERVERS` reachable from the lab network as upstreams. |
| **Procedure** | 7a: start/reuse the forwarders (`upstream_ca=CA_FILE`) and verify each forwarder's own DoT listener certificate against the **lab CA**. 7b: run the full record catalog plus the key flag/EDNS/cookie variations through every forwarder over **both** `do53` and `dot`. |
| **Expected** | Same acceptance criteria as TP-1..TP-4C; identical verdicts to the direct tier means the forwarding layer is transparent to users. |

In [ ]:
SEC = 'TP-7a forwarders up'
rows = []
FWD_TARGETS = []
try:
    import dnslab
    for name, prof in FORWARDERS:
        for inst in dnslab.start(name, profile=prof, upstreams=SERVERS,
                                 upstream_ca=CA_FILE, force=FORCE_RESTART):
            fwd = {'ip': inst.ip, 'port': 853, 'port_do53': 53,
                   'tls_hostname': inst.host, 'ca_file': dnslab.ca_file(),
                   'server': name}
            FWD_TARGETS.append(fwd)
            c = tls_cert_report(fwd)
            verdict = 'PASS' if c['ok'] else 'FAIL'
            obs = (f"{c['tls_version']} CN={c['subject_cn']}" if c['ok']
                   else c['error'])
            rows.append(record(SEC, f'TP-7a.{inst.name}', inst.host, 'dot',
                               '-', '-', f'{prof}; status={inst.status}',
                               'forwarder up; listener cert valid (lab CA)',
                               obs, verdict))
except Exception as e:  # noqa: BLE001 — no docker/dnslab: whole tier SKIPs
    rows.append(record(SEC, 'TP-7a', 'dnslab', '-', '-', '-', 'start forwarders',
                       'forwarders running', f'{type(e).__name__}: {e}', 'SKIP',
                       notes='dnslab/docker unavailable - forwarded tier skipped'))
show(rows);

In [ ]:
SEC = 'TP-7b forwarded checks'
rows = []
for fwd in FWD_TARGETS:
    label = fwd['server']
    for transport in ('do53', 'dot'):
        # record catalog
        for tid, qname, qtype in RECORD_TESTS:
            o = probe(fwd, qname, qtype, transport=transport, do=True)
            if o['error']:
                verdict = 'FAIL'
            elif o['rcode'] == 'NOERROR' and (qtype in o['answer_types']
                                              or 'CNAME' in o['answer_types']):
                verdict = 'PASS'
            else:
                verdict = 'FAIL'
            rows.append(record(SEC, f'TP-7b.{tid}.{label}.{transport}', label,
                               transport, qname, qtype, 'RD=1 DO=1 via forwarder',
                               f'NOERROR with {qtype} (or CNAME) in answer',
                               obs_str(o), verdict))
        # key variations (mirror TP-3/4/5 acceptance)
        o = probe(fwd, SIGNED_NAME, 'A', transport=transport, do=True)
        ok = (not o['error']) and o['rcode'] == 'NOERROR' \
             and ('AD' in o['flags'] or o['rrsig'])
        rows.append(record(SEC, f'TP-7b.do.{label}.{transport}', label,
                           transport, SIGNED_NAME, 'A', 'DO=1',
                           'NOERROR with AD and/or RRSIGs', obs_str(o),
                           'PASS' if ok else 'FAIL'))
        o = probe(fwd, BOGUS_NAME, 'A', transport=transport, do=True)
        ok = (not o['error']) and o['rcode'] == 'SERVFAIL'
        rows.append(record(SEC, f'TP-7b.bogus.{label}.{transport}', label,
                           transport, BOGUS_NAME, 'A', 'DO=1 on bogus zone',
                           'SERVFAIL', obs_str(o), 'PASS' if ok else 'FAIL',
                           notes='; '.join(o['ede'])))
        o = probe(fwd, NEUTRAL_NAME, 'A', transport=transport, edns=False)
        ok = (not o['error']) and o['rcode'] == 'NOERROR' and o['answers'] > 0
        rows.append(record(SEC, f'TP-7b.noedns.{label}.{transport}', label,
                           transport, NEUTRAL_NAME, 'A', 'no EDNS',
                           'NOERROR with answers', obs_str(o),
                           'PASS' if ok else 'FAIL'))
        o = probe(fwd, NEUTRAL_NAME, 'A', transport=transport, payload=512)
        ok = (not o['error']) and o['rcode'] == 'NOERROR' and o['answers'] > 0
        rows.append(record(SEC, f'TP-7b.buf512.{label}.{transport}', label,
                           transport, NEUTRAL_NAME, 'A', 'EDNS0 payload=512',
                           'NOERROR with answers', obs_str(o),
                           'PASS' if ok else 'FAIL'))
        o = probe(fwd, ECS_QNAME, 'A', transport=transport,
                  extra_options=[dns.edns.ECSOption('198.51.100.0', 24, 0)])
        ok = (not o['error']) and o['rcode'] == 'NOERROR'
        rows.append(record(SEC, f'TP-7b.ecs.{label}.{transport}', label,
                           transport, ECS_QNAME, 'A', 'ECS v4 /24 scope 0',
                           'NOERROR (ECS must not break forwarded resolution)',
                           obs_str(o), 'PASS' if ok else 'FAIL',
                           notes=f"ecs echo={o['ecs']}"))
        o = probe(fwd, NEUTRAL_NAME, 'A', transport=transport,
                  cookie=os.urandom(8))
        ok = (not o['error']) and o['rcode'] == 'NOERROR' and o['answers'] > 0
        rows.append(record(SEC, f'TP-7b.cookie.{label}.{transport}', label,
                           transport, NEUTRAL_NAME, 'A', 'client COOKIE',
                           'NOERROR with answers', obs_str(o),
                           'PASS' if ok else 'FAIL',
                           notes=f"echo={o['cookie_echoed']}"))
if not FWD_TARGETS:
    rows.append(record(SEC, 'TP-7b', '-', '-', '-', '-', '-',
                       'forwarded checks', 'no forwarders started', 'SKIP'))
df = show(rows)
print(df.groupby(['server' if 'server' in df else 'section']).size().to_dict()
      if len(df) else 'no rows')

## Results summary & export

Aggregated verdicts per section, the full failure list for follow-up, and the
complete results table exported as CSV + JSON next to this notebook
(timestamped) for the formal test record.

In [ ]:
summary = pd.DataFrame(RESULTS)
print(f'{len(summary)} checks recorded')
display(summary.pivot_table(index='section', columns='verdict',
                            values='test_id', aggfunc='count',
                            fill_value=0))

failures = summary[summary.verdict == 'FAIL']
print(f'\n{len(failures)} FAIL row(s):')
display(failures[['test_id', 'server', 'transport', 'qname', 'qtype',
                  'variation', 'expected', 'observed']])

stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')
csv_path = f'dot-test-results-{stamp}.csv'
summary.to_csv(csv_path, index=False)
summary.to_json(f'dot-test-results-{stamp}.json', orient='records', indent=1)
overall = 'FAIL' if len(failures) else 'PASS'
print(f'\nOVERALL: {overall}  (artifacts: {csv_path} / .json)')

## Report export (PDF, code-free)

Builds a formal report and renders it to PDF: the document's **markdown**
(read from the saved notebook file) plus **every result recorded in the live
session** — each TP section is followed by its table of checks, and the
summary carries the verdict pivot, failure list, and overall verdict. No code
appears in the report.

Results come straight from this kernel's `RESULTS`, so there is **no need to
save the notebook for outputs to appear** (saving only matters if you edited
the markdown text itself). Falls back to an HTML report where headless
Chromium is unavailable (e.g. running host-side).

In [ ]:
import nbformat
from nbconvert import HTMLExporter, WebPDFExporter

try:
    import ipynbname
    NB_PATH = str(ipynbname.path())
except Exception:  # noqa: BLE001 — not running under a Jupyter server
    NB_PATH = 'dot-functionality-test-plan.ipynb'   # adjust if renamed

stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')
out_base = f'dot-test-report-{stamp}'

df_all = pd.DataFrame(RESULTS)
REPORT_COLS = ['test_id', 'server', 'transport', 'qname', 'qtype',
               'variation', 'expected', 'observed', 'verdict', 'notes']


def _sec_rows(hid):
    """Rows for one heading id; TP-7 also owns TP-7a/TP-7b (lowercase
    suffix = same document section), but TP-4 does NOT own TP-4A/B/C
    (uppercase suffix = its own heading)."""
    if df_all.empty:
        return df_all
    ids = df_all.section.str.split().str[0]
    keep = (ids == hid) | (ids.str.startswith(hid)
                           & ids.str[len(hid):].str.islower())
    return df_all[keep]


def _tbl(df):
    return df[REPORT_COLS].to_html(index=False, border=0)


src_nb = nbformat.read(NB_PATH, as_version=4)
report = nbformat.v4.new_notebook()
report.cells.append(nbformat.v4.new_markdown_cell(
    '<style>table{font-size:10px;border-collapse:collapse}'
    'th,td{border:1px solid #999;padding:2px 5px;word-break:break-word;'
    'text-align:left}</style>'))

for cell in src_nb.cells:
    if cell.cell_type != 'markdown':
        continue
    first = cell.source.lstrip().splitlines()[0] if cell.source.strip() else ''
    if first.startswith('## Teardown') or first.startswith('## Report export'):
        continue
    report.cells.append(nbformat.v4.new_markdown_cell(cell.source))
    if first.startswith('## TP-'):
        hid = first.removeprefix('## ').split(' ')[0]
        rows = _sec_rows(hid)
        body = (_tbl(rows) if len(rows) else
                '_No results recorded for this section in the current session._')
        report.cells.append(nbformat.v4.new_markdown_cell(
            f'**Recorded results — {hid} ({len(rows)} checks):**\n\n' + body))
    elif first.startswith('## Results summary'):
        if df_all.empty:
            report.cells.append(nbformat.v4.new_markdown_cell(
                '_No results recorded in this session._'))
        else:
            piv = df_all.pivot_table(index='section', columns='verdict',
                                     values='test_id', aggfunc='count',
                                     fill_value=0)
            fails = df_all[df_all.verdict == 'FAIL']
            overall = 'FAIL' if len(fails) else 'PASS'
            md = (f'**Overall verdict: {overall}** — {len(df_all)} checks, '
                  f'{len(fails)} failed.\n\n' + piv.to_html(border=0))
            if len(fails):
                md += '\n\n**Failures:**\n\n' + _tbl(fails)
            report.cells.append(nbformat.v4.new_markdown_cell(md))


def _export(exporter_cls, ext, mode):
    body, _res = exporter_cls().from_notebook_node(report)
    with open(f'{out_base}.{ext}', mode) as f:
        f.write(body)
    return f'{out_base}.{ext}'


try:
    path = _export(WebPDFExporter, 'pdf', 'wb')
    print(f'report written: {path} ({len(df_all)} results from the live session)')
except Exception as e:  # noqa: BLE001 — no Chromium: fall back to HTML
    print(f'webpdf failed ({type(e).__name__}: {e}); writing HTML instead')
    print('report written:', _export(HTMLExporter, 'html', 'w'))

## Teardown (optional)

The forwarders are cheap to keep running and `dnslab.start()` reuses them on
the next run. Uncomment to remove them.

In [ ]:
# import dnslab
# dnslab.stop(all=True)      # remove the forwarder containers
# # dnslab.nuke()            # remove everything dnslab ever started